# Peaked Circuit Solver

Solve peaked quantum circuits using MPS (Matrix Product State) or general tensor network contraction.

**Only dependency: numpy** (pre-installed on Colab). Optionally uses **JAX** for TPU/GPU acceleration.

In [ ]:
# Clone the repo (safe to re-run)
%cd /content
!rm -rf peaked-solver
!git clone https://github.com/hosseinsadeghi/peaked-solver.git
%cd peaked-solver

## (Optional) Enable JAX for TPU/GPU

JAX is pre-installed on Colab. To use TPU: **Runtime > Change runtime type > TPU**.

Skip this cell to use numpy (works fine, just slower on large circuits).

In [ ]:
from peaked_solver import use_jax
use_jax(True)  # prints device info

In [ ]:
from pathlib import Path

# Discover all circuit sets
circuit_dirs = {
    'default': Path('circuits'),
    'hackathon': Path('circuits/hackathon'),
}

# Choose circuit set
CIRCUIT_SET = 'hackathon'  # @param ['default', 'hackathon']

circuit_dir = circuit_dirs[CIRCUIT_SET]
circuits = sorted(circuit_dir.glob('*.qasm'))

print(f'Circuit set: {CIRCUIT_SET} ({len(circuits)} files)\n')
print(f'{"Idx":<5} {"File":<40} {"Size":>10}')
print('-' * 58)
for i, f in enumerate(circuits):
    size_kb = f.stat().st_size / 1024
    print(f'{i:<5} {f.name:<40} {size_kb:>8.1f} KB')

## Select a circuit and inspect it

In [ ]:
from peaked_solver import QuantumCircuit
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

# ---- Pick a circuit by index ----
IDX = 0  # @param {type:"integer"}

qasm_file = circuits[IDX]
qasm_str = qasm_file.read_text()
circuit = QuantumCircuit.from_openqasm(qasm_str)

# ---- Circuit info ----
gate_counts = circuit.gate_count_by_name()
n_2q = sum(cnt for name, cnt in gate_counts.items()
           if name in ('cx', 'cz', 'cy', 'swap', 'iswap', 'ecr', 'cp', 'rzz', 'rxx', 'ryy', 'crz', 'crx', 'cry'))
n_1q = len(circuit.gates) - n_2q

print(f'File:       {qasm_file.name}')
print(f'Qubits:     {circuit.n_qubits}')
print(f'Gates:      {len(circuit.gates)}  ({n_1q} single-qubit, {n_2q} two-qubit)')
print(f'Depth:      {circuit.depth()}')
print(f'Gate types: {gate_counts}')

# ---- Extract connectivity from 2-qubit gates ----
edges = set()
for g in circuit.gates:
    if len(g.qubits) == 2:
        a, b = g.qubits
        edges.add((min(a, b), max(a, b)))

print(f'Unique 2q edges: {len(edges)}')

# ---- Draw connectivity graph ----
n = circuit.n_qubits

if n <= 200:
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))

    # Layout: try to infer grid, fall back to circular
    adj = {i: set() for i in range(n)}
    for a, b in edges:
        adj[a].add(b)
        adj[b].add(a)

    # Simple force-directed layout (spring embedding)
    rng = np.random.RandomState(42)
    pos = rng.randn(n, 2) * np.sqrt(n)
    for _ in range(300):
        # Repulsion (all pairs, capped for large n)
        disp = np.zeros((n, 2))
        for i in range(n):
            diff = pos[i] - pos
            dist = np.sqrt((diff ** 2).sum(axis=1)).clip(0.1)
            force = diff / dist[:, None] ** 2
            force[i] = 0
            disp[i] = force.sum(axis=0) * 0.5
        # Attraction (edges only)
        for a, b in edges:
            diff = pos[b] - pos[a]
            d = max(np.linalg.norm(diff), 0.1)
            f = diff * d / (n * 0.5)
            disp[a] += f
            disp[b] -= f
        # Damped update
        pos += np.clip(disp, -2, 2) * 0.3

    # Normalize
    pos -= pos.mean(axis=0)
    scale = np.abs(pos).max()
    if scale > 0:
        pos /= scale

    # Draw edges
    for a, b in edges:
        ax.plot([pos[a, 0], pos[b, 0]], [pos[a, 1], pos[b, 1]],
                'k-', alpha=0.3, linewidth=0.5)

    # Draw nodes
    node_size = max(5, min(60, 600 // n))
    ax.scatter(pos[:, 0], pos[:, 1], s=node_size, c='steelblue', zorder=5, edgecolors='white', linewidths=0.3)

    if n <= 40:
        for i in range(n):
            ax.annotate(str(i), pos[i], fontsize=max(5, 8 - n // 10),
                        ha='center', va='center', color='white', fontweight='bold')

    ax.set_title(f'{qasm_file.stem} — {n}q, {len(edges)} edges', fontsize=12)
    ax.set_aspect('equal')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f'(skipping graph for {n} qubits — too large to render)')

## Solve

In [ ]:
import time
from peaked_solver import mps_solve, tn_solve

# ---- Solver parameters ----
SOLVER    = 'mps'    # @param ['mps', '2d']
TOP_K     = 10       # @param {type:"integer"}
CHI_MAX   = 0        # @param {type:"integer"}  (0 = auto)
HEAVY_HEX = False    # @param {type:"boolean"}

# ---- Solve ----
print(f'Solving {qasm_file.name}  ({circuit.n_qubits}q, {len(circuit.gates)} gates, depth {circuit.depth()})...')
print(f'Solver: {SOLVER}, top_k: {TOP_K}, chi_max: {"auto" if CHI_MAX == 0 else CHI_MAX}, heavy_hex: {HEAVY_HEX}')
print()

t0 = time.time()

if SOLVER == 'mps':
    kwargs = {'top_k': TOP_K, 'heavy_hex': HEAVY_HEX}
    if CHI_MAX > 0:
        kwargs['chi_max'] = CHI_MAX
    result = mps_solve(circuit, **kwargs)
else:
    result = tn_solve(circuit, top_k=TOP_K, timeout=120.0)

wall = time.time() - t0

# ---- Results ----
print(f'Heuristic:  {result.heuristic_used}')
print(f'Time:       {result.compute_time_ms:.0f} ms (wall: {wall:.1f}s)')
print(f'Accuracy:   {result.accuracy_estimate:.4e}')
print(f'Trunc err:  {result.truncation_error:.2e}')
print(f'Analysis:   {result.circuit_analysis}')
print()

n = circuit.n_qubits
bitstrings = result.top_k_bitstrings
max_prob = bitstrings[0][1] if bitstrings else 0

print(f'{"Rank":<6} {"Bitstring":<{min(n+4, 64)}} {"Probability":>12}  Bar')
print('-' * (30 + min(n, 60)))
for i, (bs, prob) in enumerate(bitstrings, 1):
    bar = chr(0x2588) * int(prob / max_prob * 40) if max_prob > 0 else ''
    display = bs if len(bs) <= 56 else bs[:28] + '...' + bs[-28:]
    print(f'{i:<6} {display:<{min(n+4, 64)}} {prob:>12.6f}  {bar}')

total = sum(p for _, p in bitstrings)
print(f'\nTop-{len(bitstrings)} total probability: {total:.6f}')